## 01. Pothole / Road Damage: YOLOv8 segmentation

Trains `pothole_road_damage_model.pt` on the Roboflow Pothole Segmentation dataset (720 training and 60 validation images with polygon masks, CC BY 4.0) that ships in `ai_model/Pothole_Segmentation_YOLOv8.v1i.yolov8/`.

Our run: 60 epochs on an Apple M5 (MPS), about 41 minutes. Validation mask mAP50 0.724, precision 0.725, recall 0.711. The earlier exploratory notebook `road_damage.ipynb` reached the same 0.72 and also estimates the damaged share of the road from the masks.

In [ ]:
from pathlib import Path

# Works whether the notebook is opened from ai_model/ or ai_model/notebooks/.
ROOT = Path.cwd().resolve()
while ROOT.name != "ai_model" and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert ROOT.name == "ai_model", "Open this notebook from inside the ai_model folder"

DATA = ROOT / "data"          # downloaded and converted datasets (git-ignored)
RAW = DATA / "raw"            # original downloads
RUNS = ROOT / "runs"          # Ultralytics training runs (git-ignored)
WEIGHTS = ROOT / "weights"    # final model files used by the AI service
RESULTS = ROOT / "results"    # metrics used in the project report
for p in (DATA, RAW, RUNS, WEIGHTS, RESULTS):
    p.mkdir(parents=True, exist_ok=True)
print("ai_model root:", ROOT)

In [ ]:
import torch

# Apple Silicon uses the MPS backend; an NVIDIA GPU uses CUDA; otherwise CPU.
DEVICE = 0 if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("training device:", DEVICE)

In [ ]:
NAME = "pothole_road_damage"
src = ROOT / "Pothole_Segmentation_YOLOv8.v1i.yolov8"

# The Roboflow data.yaml points at ../train, which resolves outside the folder; write a local copy instead.
data_yaml = DATA / f"{NAME}.yaml"
data_yaml.write_text(f"path: {src}\ntrain: train/images\nval: valid/images\nnames: ['Pothole']\n")
for split in ("train", "valid"):
    print(split, len(list((src / split / "images").glob("*.jpg"))), "images")

In [ ]:
import shutil
from ultralytics import YOLO


def train(name, data_yaml, task, epochs):
    """Fine-tune a COCO-pretrained YOLOv8 nano model with the settings used for every UrbanFix category."""
    base = "yolov8n-seg.pt" if task == "segment" else "yolov8n.pt"
    model = YOLO(base)
    model.train(
        data=str(data_yaml),
        epochs=epochs,
        imgsz=640,
        batch=16,
        patience=15,          # stop if validation fitness does not improve for 15 epochs
        device=DEVICE,
        seed=0,
        deterministic=False,
        workers=4,
        plots=True,
        project=str(RUNS),
        name=name,
        exist_ok=True,
        verbose=False,
    )
    best = RUNS / name / "weights" / "best.pt"
    target = WEIGHTS / f"{name}_model.pt"
    shutil.copy(best, target)
    print("saved", target)
    return target

In [ ]:
train(NAME, data_yaml, task="segment", epochs=60)

In [ ]:
def quick_val(name, data_yaml):
    """Validate the saved model and print the headline numbers."""
    m = YOLO(str(WEIGHTS / f"{name}_model.pt"))
    r = m.val(data=str(data_yaml), imgsz=640, batch=16, device=DEVICE, plots=False, verbose=False,
              project=str(RUNS / "val"), name=name, exist_ok=True)
    print(f"box  P {r.box.mp:.3f}  R {r.box.mr:.3f}  mAP50 {r.box.map50:.3f}  mAP50-95 {r.box.map:.3f}")
    if hasattr(r, "seg") and r.seg is not None and m.task == "segment":
        print(f"mask P {r.seg.mp:.3f}  R {r.seg.mr:.3f}  mAP50 {r.seg.map50:.3f}  mAP50-95 {r.seg.map:.3f}")
    return r

In [ ]:
quick_val(NAME, data_yaml);